# EDA — Grupo 3 `Decisão` (Train Split)

Exploratory Data Analysis for Group 3 courts using `Decisão` as the target variable, restricted to the train split from the beginning.

In [ ]:
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from collections import Counter

DATA_DIR     = Path('../../data/1_raw')
PROJECT_ROOT = Path('../..').resolve()

GROUP3 = ['TRC', 'TRE', 'TRG', 'TRL', 'TRP']

In [ ]:
def field_info(df, n=3):
    rows = []
    for col in df.columns:
        raw_samples = df[col].dropna().head(n).tolist()

        cleaned_samples = []
        for v in raw_samples:
            if isinstance(v, str) and len(v) > 60:
                cleaned_samples.append(str(v)[:60] + "...")
            else:
                cleaned_samples.append(v)

        column_dict = {
            "field": col,
            "dtype": str(df[col].dtype),
            "examples": cleaned_samples,
        }
        rows.append(column_dict)

    return pd.DataFrame(rows)

def parse_decisao(obs):
    if obs is None:
        return None

    if not isinstance(obs, str):
        return None

    m = re.search(r"Decis[aã]o:\s*(.+)", obs)

    if m:
        matched_text = m.group(1)
        first_line = matched_text.split("\n")[0]
        cleaned_result = first_line.strip()
        return cleaned_result

    return None

## 1. Data Loading

### 1.1 Load the Datasets

#### TRC (Tribunal da Relação de Coimbra)

In [ ]:
with open(DATA_DIR / 'TRC.json') as f:
    trc = pd.DataFrame(json.load(f))

print(f'Records : {trc.shape[0]}')
print(f'Fields  : {trc.shape[1]}')
trc.head(2)

In [ ]:
field_info(trc)

#### TRE (Tribunal da Relação de Évora)

In [ ]:
with open(DATA_DIR / 'TRE.json') as f:
    tre = pd.DataFrame(json.load(f))

print(f'Records : {tre.shape[0]}')
print(f'Fields  : {tre.shape[1]}')
tre.head(2)

In [ ]:
field_info(tre)

#### TRG (Tribunal da Relação de Guimarães)

In [ ]:
with open(DATA_DIR / 'TRG.json') as f:
    trg = pd.DataFrame(json.load(f))

print(f'Records : {trg.shape[0]}')
print(f'Fields  : {trg.shape[1]}')
trg.head(2)

In [ ]:
field_info(trg)

#### TRL (Tribunal da Relação de Lisboa)

In [ ]:
with open(DATA_DIR / 'TRL.json') as f:
    trl = pd.DataFrame(json.load(f))

print(f'Records : {trl.shape[0]}')
print(f'Fields  : {trl.shape[1]}')
trl.head(2)

In [ ]:
field_info(trl)

#### TRP (Tribunal da Relação do Porto)

In [ ]:
with open(DATA_DIR / 'TRP.json') as f:
    trp = pd.DataFrame(json.load(f))

print(f'Records : {trp.shape[0]}')
print(f'Fields  : {trp.shape[1]}')
trp.head(2)

In [ ]:
field_info(trp)

#### Global Table

In [ ]:
summary = pd.DataFrame([
    {'source': 'TRC', 'records': trc.shape[0], 'fields': trc.shape[1]},
    {'source': 'TRE', 'records': tre.shape[0], 'fields': tre.shape[1]},
    {'source': 'TRG', 'records': trg.shape[0], 'fields': trg.shape[1]},
    {'source': 'TRL', 'records': trl.shape[0], 'fields': trl.shape[1]},
    {'source': 'TRP', 'records': trp.shape[0], 'fields': trp.shape[1]},
]).set_index('source')

print(f'Total records : {summary["records"].sum():,}')
display(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(summary.index, summary['records'], color='steelblue')
ax.set_xlabel('Tribunal')
ax.set_ylabel('Registos')
ax.set_title('Nº de registos por tribunal — Grupo 3')
for bar, val in zip(bars, summary['records']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
            f'{val:,}', ha='center', va='bottom', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### 1.2 Combined Dataset

In [ ]:
all_data = pd.concat([
    trc.assign(source='TRC'),
    tre.assign(source='TRE'),
    trg.assign(source='TRG'),
    trl.assign(source='TRL'),
    trp.assign(source='TRP'),
], ignore_index=True)

print(f'Combined records : {len(all_data):,}')
print(f'Fields           : {all_data.shape[1]}')
all_data.head(2)

## 2. Hold Out the Split

`Decisao` is parsed from `Observacoes` immediately below.

In [ ]:
all_data['Decisao'] = all_data['Observacoes'].apply(parse_decisao)

print(f'Registos com Decisao : {all_data["Decisao"].notna().sum():,} / {len(all_data):,}')
print(f'Valores únicos       : {all_data["Decisao"].nunique():,}')

In [ ]:
SPLITS_PATH = Path('../../data/2_processed/splits_grupo3.csv')

splits_df = pd.read_csv(SPLITS_PATH)[
    ['source', 'ImportJurisprudenciaId', 'Decisao', 'group_id', 'split', 'cv_fold']
].rename(columns={'Decisao': 'Decisao_split'})

before_join = len(all_data)
all_data = all_data.merge(splits_df, on=['source', 'ImportJurisprudenciaId'], how='inner')

mismatch = int((all_data['Decisao'] != all_data['Decisao_split']).sum())
print(f'Decisao mismatches vs. 01_Data_Split\'s export : {mismatch:,}')
assert mismatch == 0, (
    'parse_decisao has drifted between 04_Data_Split and this notebook -- the two must '
    'extract identical Decisao values for the join to be valid.'
)
all_data = all_data.drop(columns='Decisao_split')

print(f'Records before join (raw, this notebook) : {before_join:,}')
print(f'Records after join  (01 \u2229 this notebook) : {len(all_data):,}')
print(f'Dropped (unlabelled, or below 01\'s rare-class floor) : {before_join - len(all_data):,}')

all_data = all_data[all_data['split'] == 'train'].reset_index(drop=True)
total = len(all_data)
print(f'\nRestricted to train split : {total:,} records')


## 3. Data Quality Assessment

#### Missing Variables' Values

In [ ]:
total = len(all_data)

missing = pd.DataFrame({
    'null_count':  all_data.isnull().sum(),
    'empty_count': (all_data == '').sum(),
}).assign(
    total_missing=lambda df: df['null_count'] + df['empty_count'],
    pct=lambda df: (df['total_missing'] / total * 100).round(2),
).sort_values('total_missing', ascending=False)

missing[missing['total_missing'] > 0]

#### Sumario — missing breakdown by source

In [ ]:
sumario_missing = (
    all_data.assign(missing=all_data['Sumario'].isna() | (all_data['Sumario'] == ''))
    .groupby('source')
    .agg(total=('missing', 'count'), missing=('missing', 'sum'))
    .assign(
        present=lambda d: d['total'] - d['missing'],
        pct_missing=lambda d: (d['missing'] / d['total'] * 100).round(1),
    )
)
display(sumario_missing)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = sumario_missing.index
width = 0.4

ax.bar(x, sumario_missing['total'],   width, label='Total',   color='steelblue')
ax.bar(x, sumario_missing['missing'], width, label='Missing', color='tomato')
ax.set_xlabel('Tribunal')
ax.set_ylabel('Registos')
ax.set_title('Sumario — total vs. missing por tribunal')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### 3.2 Duplicate Records

Four levels of duplication are checked.

In [ ]:
dup_process = all_data[all_data.duplicated('NumeroProcesso', keep=False)].shape[0]
dup_sumario = all_data[all_data['Sumario'].notna()
                       & all_data.duplicated('Sumario', keep=False)].shape[0]

def normalise(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

sumario_norm  = all_data['Sumario'].dropna().apply(normalise)
near_dup_mask = sumario_norm.duplicated(keep=False)
dup_near      = near_dup_mask.sum()

pd.DataFrame({
    'check': [
        'Same NumeroProcesso',
        'Same Sumario (exact)',
        'Same Sumario (normalised)',
    ],
    'affected_records': [dup_process, dup_sumario, dup_near],
    'pct': [
        round(dup_process / total * 100, 2),
        round(dup_sumario / total * 100, 2),
        round(dup_near    / total * 100, 2),
    ],
})

In [ ]:
exact_by_source = (
    all_data[all_data['Sumario'].notna() & all_data.duplicated('Sumario', keep=False)]
    .groupby('source').size().rename('exato')
)
near_dup_rows = all_data.loc[sumario_norm[near_dup_mask].index,
                             ['source', 'NumeroProcesso', 'Sumario']].copy()
near_by_source = near_dup_rows.groupby('source').size().rename('normalizado')

dup_by_source = pd.concat([exact_by_source, near_by_source], axis=1).fillna(0).astype(int)
display(dup_by_source)

#### 3.2.1 Impacto da Normalização (lowercase, pontuação, lematização)

Compara o número de registos afectados por duplicação em cada etapa de normalização do `Sumario`.

In [ ]:
import spacy

nlp_lemma = spacy.load('pt_core_news_sm', disable=['parser', 'ner'])

def lemmatise(doc):
    return ' '.join(tok.lemma_.lower() for tok in doc if not tok.is_space)

norm_texts    = sumario_norm.str.slice(0, 3000)
lemma_docs    = nlp_lemma.pipe(norm_texts.tolist(), batch_size=64)
sumario_lemma = pd.Series([lemmatise(d) for d in lemma_docs], index=norm_texts.index)

dup_lemma_mask = sumario_lemma.duplicated(keep=False)
dup_lemma      = int(dup_lemma_mask.sum())

normalization_impact = pd.DataFrame({
    'stage': [
        'Original (exato)',
        '+ lowercase + pontuação removida',
        '+ lematização',
    ],
    'affected_records': [dup_sumario, dup_near, dup_lemma],
})
normalization_impact['pct'] = (normalization_impact['affected_records'] / total * 100).round(2)
normalization_impact['incremental_vs_previous'] = normalization_impact['affected_records'].diff()
normalization_impact

#### 3.2.2 Quase-Duplicados via Distância de Edição Mínima (Levenshtein)

Para além dos duplicados idênticos após normalização.

In [ ]:
from rapidfuzz import fuzz, process

RATIO_THRESHOLD = 90

candidates   = sumario_norm[~near_dup_mask]
cand_source  = all_data.loc[candidates.index, 'source']
cand_bucket  = candidates.str.len() // 50

blocks = pd.DataFrame({'text': candidates, 'source': cand_source, 'bucket': cand_bucket})

near_edit_pairs = []
for _, grp in blocks.groupby(['source', 'bucket']):
    if len(grp) < 2:
        continue
    idx   = grp.index.to_list()
    texts = grp['text'].to_list()
    mat   = process.cdist(texts, texts, scorer=fuzz.ratio, score_cutoff=RATIO_THRESHOLD, workers=-1)
    for i in range(len(idx)):
        for j in range(i + 1, len(idx)):
            score = mat[i, j]
            if score >= RATIO_THRESHOLD:
                near_edit_pairs.append((idx[i], idx[j], float(score)))

near_edit_idx = sorted({i for pair in near_edit_pairs for i in pair[:2]})
dup_edit      = len(near_edit_idx)

print(f'Pares quase-duplicados (Levenshtein, ratio >= {RATIO_THRESHOLD}) : {len(near_edit_pairs):,}')
print(f'Registos afectados                                           : {dup_edit:,} ({dup_edit / total * 100:.2f}%)')

In [ ]:
pd.DataFrame(
    [
        (
            all_data.loc[i, 'source'], all_data.loc[i, 'NumeroProcesso'],
            all_data.loc[j, 'source'], all_data.loc[j, 'NumeroProcesso'],
            round(score, 1),
        )
        for i, j, score in sorted(near_edit_pairs, key=lambda x: -x[2])[:10]
    ],
    columns=['source_a', 'processo_a', 'source_b', 'processo_b', 'ratio'],
)

#### 3.2.3 Resumo Consolidado

In [ ]:
summary_all = pd.DataFrame({
    'check': [
        'Same NumeroProcesso',
        'Same Sumario (exact)',
        'Same Sumario (lowercase + pontuação)',
        'Same Sumario (+ lematização)',
        'Quase-duplicado (Levenshtein, ratio >= 90)',
    ],
    'affected_records': [dup_process, dup_sumario, dup_near, dup_lemma, dup_edit],
})
summary_all['pct'] = (summary_all['affected_records'] / total * 100).round(2)
summary_all


### 3.3 Malformed / Placeholder Sumario

Checks run on the train-only frame directly.

In [ ]:
PLACEHOLDER_PATTERNS = [
    (r'^\s*$',                       'só espaços'),
    (r'^\s*[.\-_/\\|]{1,3}\s*$',    'só pontuação'),
    (r'^\s*n\.?/?a\.?\s*$',          'N/A'),
    (r'^\s*na[oã]?\s+h[aá]\s*',      'sem sumário (texto)'),
    (r'^\s*sem\s+sum[aá]rio\s*$',    'sem sumário'),
    (r'^\s*.{1,3}\s*$',              '<= 3 caracteres'),
    (r'^\s*.{4,9}\s*$',              '4-9 caracteres'),
]

s = all_data['Sumario'].dropna()
results = []
for pattern, label in PLACEHOLDER_PATTERNS:
    mask = s.str.match(pattern, case=False)
    results.append({
        'categoria': label,
        'padrao':    pattern,
        'registos':  mask.sum(),
        'pct':       round(mask.sum() / total * 100, 3),
    })

pd.DataFrame(results)

#### 3.4 Sumarios suspeitos — menos de 20 tokens

In [ ]:
TOKEN_THRESHOLD = 20

token_counts = all_data['Sumario'].dropna().str.split().str.len()
short_flag   = token_counts < TOKEN_THRESHOLD

flagged = all_data.loc[short_flag[short_flag].index,
                       ['source', 'NumeroProcesso', 'Sumario']].copy()
flagged['token_count'] = token_counts[short_flag].values

print(f'Sumarios com < {TOKEN_THRESHOLD} tokens : {len(flagged):,} ({len(flagged)/total*100:.1f}%)')
display(flagged.sort_values('token_count').head(10)[['source', 'NumeroProcesso', 'token_count', 'Sumario']])

In [ ]:
short_by_source = flagged.groupby('source').size()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(short_by_source.index, short_by_source.values, color='steelblue')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Tribunal')
ax.set_ylabel('Registos')
ax.set_title(f'Sumarios com < {TOKEN_THRESHOLD} tokens por tribunal')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Target Variable Analysis — `Decisão`

### 4.1 Cobertura por Tribunal

In [ ]:
coverage = (
    all_data.groupby('source')
    .agg(
        total=('Decisao', 'count'),
        com_decisao=('Decisao', lambda x: x.notna().sum()),
    )
    .assign(
        sem_decisao=lambda d: d['total'] - d['com_decisao'],
        pct_cobertura=lambda d: (d['com_decisao'] / d['total'] * 100).round(1),
    )
)
display(coverage)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(coverage.index, coverage['com_decisao'],  color='steelblue', label='Com Decisão')
axes[0].bar(coverage.index, coverage['sem_decisao'],  color='lightgrey', label='Sem Decisão',
            bottom=coverage['com_decisao'])
axes[0].set_title('Cobertura absoluta por tribunal')
axes[0].set_xlabel('Tribunal')
axes[0].set_ylabel('Registos')
axes[0].legend()
axes[0].spines[['top', 'right']].set_visible(False)

bars = axes[1].bar(coverage.index, coverage['pct_cobertura'], color='steelblue')
for bar, val in zip(bars, coverage['pct_cobertura']):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
axes[1].set_title('% registos com Decisão por tribunal')
axes[1].set_xlabel('Tribunal')
axes[1].set_ylabel('%')
axes[1].set_ylim(0, 115)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

Cobertura heterogénea.

### 4.2 Top 20 Classes — Grupo 3 (global)

In [ ]:
class_counts = all_data['Decisao'].value_counts()
n_classes    = class_counts.shape[0]
n_with       = all_data['Decisao'].notna().sum()

print(f'Número de classes (Decisao) : {n_classes:,}')
print(f'Total de registos com Decisao : {n_with:,}')
print()
print('Top 20:')
display(class_counts.head(20).rename('count').to_frame()
        .assign(pct=lambda d: (d['count'] / n_with * 100).round(2)))

In [ ]:
top20 = class_counts.head(20).sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

bars = ax1.barh(top20.index, top20.values, color='steelblue', alpha=0.85, edgecolor='white')
for bar, val in zip(bars, top20.values):
    ax1.text(bar.get_width() + top20.values.max() * 0.01,
             bar.get_y() + bar.get_height() / 2,
             f'{val:,}', va='center', ha='left', fontsize=8.5)
ax1.set_title('Top 20 Decisão — contagem absoluta')
ax1.set_xlabel('Nº de ocorrências')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.set_xlim(right=top20.values.max() * 1.18)
ax1.spines[['top', 'right']].set_visible(False)

top20_pct = (top20 / n_with * 100)
bars2 = ax2.barh(top20_pct.index, top20_pct.values, color='#55A868', alpha=0.85, edgecolor='white')
for bar, val in zip(bars2, top20_pct.values):
    ax2.text(bar.get_width() + top20_pct.values.max() * 0.01,
             bar.get_y() + bar.get_height() / 2,
             f'{val:.1f}%', va='center', ha='left', fontsize=8.5)
ax2.set_title('Top 20 Decisão — % do total com Decisão')
ax2.set_xlabel('% de registos com Decisão')
ax2.set_xlim(right=top20_pct.values.max() * 1.18)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### 4.3 Top 20 Classes — distribuição por tribunal

In [ ]:
top20_labels = class_counts.head(20).index.tolist()

court_series = {
    court: all_data[all_data['source'] == court]['Decisao'].value_counts()
    for court in GROUP3
}

df_top20 = pd.DataFrame(
    {court: court_series[court] for court in GROUP3},
    index=top20_labels,
).fillna(0).astype(int)

display(df_top20)

In [ ]:
COURT_COLORS = ['#d73027', '#f46d43', '#fdae61', '#a5c4e0', '#c7a0d8']

sorted_labels = df_top20.sum(axis=1).sort_values().index.tolist()
df_plot = df_top20.loc[sorted_labels]

fig, ax = plt.subplots(figsize=(14, 9))
lefts = np.zeros(len(sorted_labels))

for court, color in zip(GROUP3, COURT_COLORS):
    vals = df_plot[court].values
    ax.barh(sorted_labels, vals, left=lefts, color=color, alpha=0.88,
            edgecolor='white', linewidth=0.5, label=court)
    lefts += vals

totals = df_plot.sum(axis=1).values
for i, total in enumerate(totals):
    ax.text(total + totals.max() * 0.01, i, f'{int(total):,}',
            va='center', ha='left', fontsize=8.5, color='#333333')

ax.set_title('Top 20 Decisão — distribuição por tribunal (Grupo 3)', fontsize=12, fontweight='bold')
ax.set_xlabel('Nº de ocorrências')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_xlim(right=totals.max() * 1.18)
ax.tick_params(axis='y', labelsize=9)
ax.legend(fontsize=9, loc='lower right', framealpha=0.7)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### 4.4 Top 20 por tribunal individual

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(30, 9))
fig.suptitle('Top 20 Decisão — por tribunal individual (Grupo 3)', fontsize=13, fontweight='bold', y=1.01)

for ax, court, color in zip(axes, GROUP3, COURT_COLORS):
    vals = all_data[all_data['source'] == court]['Decisao'].dropna()
    if len(vals) > 0:
        series = vals.value_counts().head(20).sort_values()
        bars = ax.barh(series.index, series.values, color=color, alpha=0.88, edgecolor='white')
        for bar, val in zip(bars, series.values):
            ax.text(bar.get_width() + series.values.max() * 0.02,
                    bar.get_y() + bar.get_height() / 2,
                    f'{int(val):,}', va='center', ha='left', fontsize=7.5, color='#333333')
        ax.set_xlim(right=series.values.max() * 1.25)
    else:
        ax.text(0.5, 0.5, 'Sem dados', transform=ax.transAxes,
                ha='center', va='center', fontsize=9, color='grey')

    n_total   = len(all_data[all_data['source'] == court])
    n_labeled = len(vals)
    ax.set_title(f'{court}\n({n_labeled:,}/{n_total:,} = {n_labeled/n_total*100:.0f}%)',
                 fontsize=9, fontweight='bold', pad=6)
    ax.set_xlabel('Nº ocorrências', fontsize=8)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='y', labelsize=7.5)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### 4.5 Imbalance Summary

In [ ]:
top20_counts = class_counts.head(20)

imbalance = top20_counts.to_frame('count').assign(
    pct=lambda d: (d['count'] / n_with * 100).round(2),
    imbalance_ratio=lambda d: (top20_counts.max() / d['count']).round(1),
)
print(f'Majority class : {top20_counts.idxmax()}  ({top20_counts.max():,} registos)')
print(f'Minority class (top 20) : {top20_counts.iloc[19]}  registos')
print(f'Imbalance ratio (max/min top20) : {top20_counts.max() / top20_counts.iloc[19]:.1f}x')
print()
display(imbalance)

## 5. Sumario Text Analysis

### 5.1 Length & Token-Count Distributions

Metrics computed on all non-null `Sumario` entries: - `char_len`.

In [ ]:
import sys
from transformers import AutoTokenizer

sys.path.insert(0, str(PROJECT_ROOT))
from src.config import MODEL_REGISTRY

df_len = all_data[all_data['Sumario'].notna()].copy()
df_len['char_len']   = df_len['Sumario'].str.len()
df_len['word_count'] = df_len['Sumario'].str.split().str.len()

token_count_cols = {}
over_limit_masks = {}

for slug, cfg in MODEL_REGISTRY.items():
    print(f'Loading tokenizer: {cfg.hf_id}  (revision={cfg.revision}) ...')
    tok = AutoTokenizer.from_pretrained(cfg.hf_id, revision=cfg.revision)
    encodings = tok(
        df_len['Sumario'].tolist(),
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_attention_mask=False,
    )
    col = f'token_count_{slug}'
    df_len[col] = [len(ids) for ids in encodings['input_ids']]
    token_count_cols[slug] = col
    over_limit_masks[slug] = df_len[col] > cfg.max_length

print(f'\nEntries analysed : {len(df_len):,}')
print('Tokenization complete for all models in MODEL_REGISTRY.')

In [ ]:
summary_rows = []
for slug, cfg in MODEL_REGISTRY.items():
    col  = token_count_cols[slug]
    mask = over_limit_masks[slug]
    summary_rows.append({
        'model':            cfg.display_name,
        'max_length':       cfg.max_length,
        'n_over_limit':     int(mask.sum()),
        'pct_over_limit':   round(mask.mean() * 100, 2),
        'max_token_len':    int(df_len[col].max()),
        'median_token_len': int(df_len[col].median()),
    })

summary_df = pd.DataFrame(summary_rows).set_index('model')
display(summary_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

cols = [
    ('char_len',   'Character length', None),
    ('word_count', 'Word count',       None),
]

for ax, (col, title, threshold) in zip(axes, cols):
    data_plot = df_len[col].clip(upper=df_len[col].quantile(0.99))
    ax.hist(data_plot, bins=60, color='steelblue', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
models_plot = list(summary_df.index)
pcts_plot   = summary_df['pct_over_limit'].tolist()

bars = ax.bar(models_plot, pcts_plot, color='#C0392B', alpha=0.85, edgecolor='white')
for bar, val in zip(bars, pcts_plot):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_title("Records exceeding each model's max_length, by tokenizer", fontsize=12, fontweight='bold')
ax.set_ylabel('% of records')
ax.set_xticks(range(len(models_plot)))
ax.set_xticklabels(models_plot, rotation=30, ha='right')
ax.set_ylim(0, max(pcts_plot) * 1.25)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
stats_global = (
    df_len[['char_len', 'word_count']]
    .agg(['mean', 'median', 'min', 'max', 'std'])
    .round(1)
    .rename(index={'mean': 'Mean', 'median': 'Median', 'min': 'Min', 'max': 'Max', 'std': 'Std'})
)
display(stats_global)

In [ ]:
top10_classes = class_counts.head(10).index.tolist()
has_decisao   = df_len[df_len['Decisao'].isin(top10_classes)]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

col_pairs = [
    ('char_len',   'Character length', None),
    ('word_count', 'Word count',       None),
]

for ax, (col, title, threshold) in zip(axes, col_pairs):
    for cls in top10_classes:
        sub = has_decisao[has_decisao['Decisao'] == cls][col]
        ax.hist(sub.clip(upper=sub.quantile(0.99)), bins=40, alpha=0.35, label=cls)
    ax.set_title(f'{title} — por classe (top 10)')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend(fontsize=6, loc='upper right')
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### 5.2 Vocabulary Analysis

In [ ]:
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

nlp          = spacy.load('pt_core_news_sm', disable=['parser', 'ner', 'lemmatizer'])
PT_STOPWORDS = nlp.Defaults.stop_words

def tokenize(text):
    tokens = re.sub(r'[^\w\s]', ' ', text.lower()).split()
    return [t for t in tokens if t not in PT_STOPWORDS and len(t) > 2 and not t.isdigit()]

def tokenize_str(text):
    return ' '.join(tokenize(text))

In [ ]:
all_tokens = []
for text in df_len['Sumario']:
    all_tokens.extend(tokenize(text))

token_freq = Counter(all_tokens)

total_tok = len(all_tokens)
cumsum, coverage_vocab = 0, 0
for count in sorted(token_freq.values(), reverse=True):
    cumsum += count
    coverage_vocab += 1
    if cumsum / total_tok >= 0.80:
        break

print(f'Vocabulary size       : {len(token_freq):,} unique tokens')
print(f'Total tokens          : {total_tok:,}')
print(f'Top-{coverage_vocab} tokens cover 80% of corpus')

In [ ]:
top_n = 30
terms, counts = zip(*token_freq.most_common(top_n))

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(terms[::-1], counts[::-1], color='steelblue')
ax.set_xlabel('Frequency')
ax.set_title(f'Top {top_n} tokens por frequência (stopwords removidas)')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
doc_freq = Counter()
for text in df_len['Sumario']:
    doc_freq.update(set(tokenize(text)))

top_n = 30
df_terms, df_counts = zip(*doc_freq.most_common(top_n))

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(df_terms[::-1], df_counts[::-1], color='steelblue')
ax.set_xlabel('Document frequency (nº de documentos)')
ax.set_title(f'Top {top_n} tokens por document frequency')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
TOP_TERMS = 15
tfidf_rows = []

for cls in top10_classes:
    docs = (
        df_len.loc[df_len['Decisao'] == cls, 'Sumario']
        .dropna()
        .apply(tokenize_str)
    )
    if len(docs) < 2:
        continue
    vec   = TfidfVectorizer(max_features=5000)
    X     = vec.fit_transform(docs)
    means = X.mean(axis=0).A1
    top_i = means.argsort()[::-1][:TOP_TERMS]
    for rank, i in enumerate(top_i, 1):
        tfidf_rows.append({
            'class': cls,
            'rank':  rank,
            'term':  vec.get_feature_names_out()[i],
            'tfidf': round(float(means[i]), 4),
        })

tfidf_df = pd.DataFrame(tfidf_rows)
display(tfidf_df.pivot(index='rank', columns='class', values='term'))

In [ ]:
corpus = df_len['Sumario'].str.lower().str.replace(r'[^\w\s]', ' ', regex=True).tolist()
labels = df_len['Decisao'].tolist()

top10_set = set(top10_classes)

vec = CountVectorizer(ngram_range=(3, 5), min_df=0.02, max_features=100_000)
X   = vec.fit_transform(corpus)
features = vec.get_feature_names_out()

class_phrases = {}
for cls in top10_classes:
    idx      = [i for i, l in enumerate(labels) if l == cls]
    X_cls    = X[idx]
    doc_freq = (X_cls > 0).mean(axis=0).A1
    class_phrases[cls] = set(features[doc_freq >= 0.30])

boilerplate = sorted(set.intersection(*class_phrases.values()))
print(f'Frases boilerplate (>=30% em cada classe, top 10): {len(boilerplate)}')
print()

coverage_cls = {cls: f'{len(class_phrases[cls]):,}' for cls in top10_classes}
print('Frases que atingem 30% por classe:')
for cls, n in coverage_cls.items():
    print(f'  {cls:<40} {n}')
print()
pd.Series(boilerplate, name='boilerplate_phrase')

### 5.3 Linguistic Patterns

Checks language consistency, detects mixed-language or corrupted entries, and measures structural complexity of the `Sumario` field.

#### Language Detection

In [ ]:
from langdetect import detect, LangDetectException

def detect_lang(text):
    try:
        return detect(text)
    except LangDetectException:
        return 'error'

df_lang = df_len[['Sumario', 'source', 'Decisao']].copy()
df_lang['lang'] = df_lang['Sumario'].apply(detect_lang)

lang_counts = df_lang['lang'].value_counts()
print('Language distribution:')
print(lang_counts.to_string())
print(f"\nPortuguese (pt): {lang_counts.get('pt', 0) / len(df_lang) * 100:.1f}% of entries")

#### Mixed-Language and Corrupted Entries

In [ ]:
non_pt = df_lang[df_lang['lang'] != 'pt'].copy()
print(f'Non-Portuguese entries: {len(non_pt)} ({len(non_pt)/len(df_lang)*100:.1f}%)')
print()

df_lang['alpha_ratio'] = df_lang['Sumario'].str.replace(r'\s', '', regex=True) \
    .apply(lambda s: sum(c.isalpha() for c in s) / max(len(s), 1))
corrupted = df_lang[df_lang['alpha_ratio'] < 0.5]
print(f'Corrupted entries (alpha_ratio < 0.5): {len(corrupted)} ({len(corrupted)/len(df_lang)*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

lang_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribuição de idiomas detectados')
axes[0].set_xlabel('Language code')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].hist(df_lang['alpha_ratio'], bins=40, color='steelblue', edgecolor='white')
axes[1].axvline(0.5, color='tomato', linestyle='--', linewidth=1.2, label='limiar 0.5')
axes[1].set_title('Distribuição do alpha_ratio')
axes[1].set_xlabel('Proporção de caracteres alfabéticos')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

#### Sentence Length and Structural Complexity

In [ ]:
nlp_sent = spacy.load('pt_core_news_sm', disable=['ner', 'lemmatizer'])
if 'sentencizer' not in nlp_sent.pipe_names and 'parser' not in nlp_sent.pipe_names:
    nlp_sent.add_pipe('sentencizer')

df_len = df_len.drop(columns=[c for c in ['n_sentences', 'avg_sent_len', 'max_sent_len', 'std_sent_len']
                               if c in df_len.columns])

all_sent_lens = []
doc_n_sents   = []
doc_std_lens  = []

for text in df_len['Sumario']:
    doc  = nlp_sent(text[:10_000])
    lens = [len(s.text.split()) for s in doc.sents]
    all_sent_lens.extend(lens)
    doc_n_sents.append(len(lens))
    doc_std_lens.append(pd.Series(lens).std() if len(lens) > 1 else 0)

df_len['n_sentences']  = doc_n_sents
df_len['std_sent_len'] = doc_std_lens

sent_lens = pd.Series(all_sent_lens, name='words_per_sentence')

print('--- Sentence length distribution (all sentences flattened) ---')
display(sent_lens.describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(sent_lens.clip(upper=300).to_numpy(), bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Sentence length (words) — clipped at 300')
axes[0].set_xlabel('Words per sentence')
axes[0].set_ylabel('Count')
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].hist(df_len['n_sentences'].clip(upper=50), bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Nº de frases por documento — clipped at 50')
axes[1].set_xlabel('Nº de frases')
axes[1].set_ylabel('Count')
axes[1].spines[['top', 'right']].set_visible(False)

axes[2].hist(df_len['std_sent_len'].clip(upper=100), bins=40, color='steelblue', edgecolor='white')
axes[2].set_title('Std da comprimento de frase por doc')
axes[2].set_xlabel('Desvio padrão (palavras)')
axes[2].set_ylabel('Count')
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
complexity = (
    df_len[df_len['Decisao'].isin(top10_classes)]
    .groupby('Decisao')[['n_sentences', 'std_sent_len']]
    .agg(['mean', 'median'])
    .round(2)
    .sort_values(('n_sentences', 'mean'), ascending=False)
)
print('--- Document complexity by Decisao class (top 10) ---')
display(complexity)

## 6. Inter-Class Analysis

Measures how lexically similar different `Decisão` classes are.

In [ ]:
from itertools import combinations

TOP_N        = 20
top_classes  = class_counts.head(TOP_N).index.tolist()

class_vocab = {}
for cls in top_classes:
    texts = df_len[df_len['Decisao'] == cls]['Sumario'].dropna()
    words = set()
    for t in texts:
        words.update(tokenize(t))
    class_vocab[cls] = words

overlap_rows = []
for c1, c2 in combinations(top_classes, 2):
    v1, v2 = class_vocab[c1], class_vocab[c2]
    inter  = len(v1 & v2)
    union  = len(v1 | v2)
    overlap_rows.append({
        'class_a': c1, 'class_b': c2,
        'jaccard': round(inter / union if union else 0.0, 4),
        'shared_tokens': inter,
        'vocab_a': len(v1),
        'vocab_b': len(v2),
    })

overlap_df = (pd.DataFrame(overlap_rows)
                .sort_values('jaccard', ascending=False)
                .reset_index(drop=True))

print("Top 15 most similar class pairs (Jaccard vocabulary overlap):")
display(overlap_df.head(15))

In [ ]:
import seaborn as sns

jaccard_matrix = pd.DataFrame(
    np.ones((TOP_N, TOP_N)), index=top_classes, columns=top_classes
)
for _, row in overlap_df.iterrows():
    jaccard_matrix.loc[row['class_a'], row['class_b']] = row['jaccard']
    jaccard_matrix.loc[row['class_b'], row['class_a']] = row['jaccard']

short_labels = [c[:28] for c in top_classes]

fig, ax = plt.subplots(figsize=(15, 12))
sns.heatmap(
    jaccard_matrix.values,
    xticklabels=short_labels, yticklabels=short_labels,
    annot=True, fmt='.2f', cmap='YlOrRd',
    ax=ax, linewidths=0.3, vmin=0, vmax=1,
    annot_kws={'size': 7},
)
ax.set_title('Pairwise Jaccard Lexical Overlap — Top 20 Decisão classes (Grupo 3)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0,  fontsize=8)
plt.tight_layout()
plt.show()

### 6.1 Hardest-to-Distinguish Pairs

Pairs with Jaccard > 0.6 share the majority of their vocabulary.

### 6.2 2D Class Projection (TF-IDF centroid + UMAP / PCA)

Each class is represented by the mean TF-IDF vector of its documents.

In [ ]:
try:
    import umap
    USE_UMAP = True
except ImportError:
    USE_UMAP = False
    print("umap-learn not installed — falling back to PCA")

tfidf_ic = TfidfVectorizer(max_features=8000, sublinear_tf=True,
                            tokenizer=tokenize, token_pattern=None)
corpus_top20 = df_len[df_len['Decisao'].isin(top_classes)].copy()
tfidf_ic.fit(corpus_top20['Sumario'].fillna(''))

centroids, valid_classes = [], []
for cls in top_classes:
    texts = df_len[df_len['Decisao'] == cls]['Sumario'].fillna('')
    if len(texts) == 0:
        continue
    mat = tfidf_ic.transform(texts)
    centroids.append(np.asarray(mat.mean(axis=0)).flatten())
    valid_classes.append(cls)

centroid_matrix = np.stack(centroids)

if USE_UMAP:
    reducer = umap.UMAP(n_components=2, random_state=42,
                        n_neighbors=min(5, len(valid_classes) - 1), min_dist=0.3)
    coords = reducer.fit_transform(centroid_matrix)
    method = 'UMAP'
else:
    from sklearn.decomposition import PCA
    coords = PCA(n_components=2, random_state=42).fit_transform(centroid_matrix)
    method = 'PCA'

sizes  = [class_counts[c] / class_counts.max() * 500 + 60 for c in valid_classes]
colors = plt.cm.tab20(np.linspace(0, 1, len(valid_classes)))

fig, ax = plt.subplots(figsize=(13, 10))
for i, cls in enumerate(valid_classes):
    x, y = coords[i]
    ax.scatter(x, y, s=sizes[i], color=colors[i], alpha=0.85,
               edgecolors='white', linewidth=0.5)
    ax.annotate(cls[:32], (x, y), fontsize=8, ha='center', va='bottom',
                xytext=(0, 6), textcoords='offset points')

ax.set_title(
    f'TF-IDF Centroid Projection ({method}) — Top 20 Decisão · Grupo 3\n'
    '(marker size ∝ class frequency)',
    fontsize=12
)
ax.set_xlabel(f'{method} Dim 1')
ax.set_ylabel(f'{method} Dim 2')
plt.tight_layout()
plt.show()

## 7. Label Leakage Warning

If the `Sumario` text explicitly states the outcome (e.g., *'recurso improcedente'*, *'confirmada a decisão'*).

In [ ]:
LEAKAGE_PATTERNS = [
    r'recurso improcedente',
    r'negado provimento',
    r'negar provimento',
    r'nao provido',
    r'apelacao improcedente',
    r'apelação improcedente',
    r'confirma(?:da|ndo|r)',
    r'revog(?:ada|ando|ar)\b',
    r'recurso procedente',
    r'parcialmente procedente',
    r'provimento ao recurso',
]

leakage_pattern = re.compile('|'.join(LEAKAGE_PATTERNS), re.IGNORECASE)

has_sumario = all_data[all_data['Sumario'].notna()].copy()
leakage_mask = has_sumario['Sumario'].str.contains(leakage_pattern, na=False)

n_leak = leakage_mask.sum()
pct    = n_leak / len(has_sumario) * 100

print(f'Registos com frases reveladoras de outcome : {n_leak:,}  ({pct:.1f}%)')
print()

leak_by_source = (
    has_sumario[leakage_mask].groupby('source').size()
    .rename('com_leakage')
    .to_frame()
    .join(has_sumario.groupby('source').size().rename('total'))
    .assign(pct=lambda d: (d['com_leakage'] / d['total'] * 100).round(1))
)
display(leak_by_source)

In [ ]:
has_sumario[leakage_mask][['source', 'Decisao', 'Sumario']].head(5) \
    .assign(Sumario=lambda d: d['Sumario'].str[:200])

## 8. Key Findings

## 9. Summary and Modeling Recommendations

Consolidates all EDA findings into actionable decisions for the modeling phase.